# 18. Model Optimization for Deployment


## 1. Load the existing model

**Explanation:** We need a model to inspect. This project already has a fine-tuned ResNet-18 checkpoint. The code loads its weights into the matching 10-class model and selects a device available on this computer.

**Example code:**

In [1]:
from pathlib import Path
import time
import torch
from torchvision import models
checkpoint = Path("artifacts/resnet18_cifar10_finetuned.pth")
if not checkpoint.is_file():
    raise FileNotFoundError("Run notebook 10 first to create the ResNet-18 checkpoint.")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, 10)
model.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True))
model = model.to(device).eval()
print("Model loaded on:", device)

Model loaded on: cpu


**Interpretation:** We are inspecting a model trained in an earlier lesson. `eval()` switches it to inference behavior; it does not train or change the learned weights.

## 2. Parameter count

**Explanation:** Parameters are the learned numbers stored in a model. More parameters often mean more computation and storage, but parameter count alone does not determine accuracy or speed.

**Example:** Count the model's learned parameters.

In [2]:
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Parameters: {parameter_count:,}")

Parameters: 11,181,642


**Interpretation:** This count tells us how many learned values are in this ResNet-18. It is one useful model-size clue, not a direct measurement of its file size or latency.

## 3. Model size

**Explanation:** Model size on disk is how much storage a saved model needs. A rough estimate for float32 weights is four bytes per parameter, but the checkpoint can also include other data or use another format.

**Example:** Check the actual size of the saved weights.

In [3]:
size_mb = checkpoint.stat().st_size / (1024 ** 2)
rough_float32_mb = parameter_count * 4 / (1024 ** 2)
print(f"Checkpoint on disk: {size_mb:.2f} MB")
print(f"Approx. float32 parameter storage: {rough_float32_mb:.2f} MB")

Checkpoint on disk: 42.73 MB
Approx. float32 parameter storage: 42.65 MB


**Interpretation:** The first number is the actual checkpoint file size. The second is a rough estimate for parameters only; the two can differ because saved files contain metadata and may use different storage formats.

## 4. Inference time

**Explanation:** Inference is using a trained model to make predictions. Inference time is how long that prediction takes. A deployment should measure representative inputs on its target device.

**Example:** Time a few forward passes with one 96 × 96 image. Warm-up passes are excluded because the first call can include setup overhead.

In [4]:
sample = torch.rand(1, 3, 96, 96, device=device)
with torch.inference_mode():
    for _ in range(3):
        model(sample)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(10):
        model(sample)
    if device.type == "cuda":
        torch.cuda.synchronize()
    average_ms = (time.perf_counter() - start) * 1000 / 10
print(f"Average time per image on {device}: {average_ms:.2f} ms")

Average time per image on cpu: 27.86 ms


**Interpretation:** This is an approximate local timing for the chosen model, image size, and device. Try more repetitions and the actual deployment hardware for a reliable comparison.

## 5. Batch inference

**Explanation:** Batch inference predicts several images in one model call. It can process images more efficiently, but it needs more memory and may add waiting time while a batch fills.

**Example:** Send four images at once.

In [5]:
batch = torch.rand(4, 3, 96, 96, device=device)
with torch.inference_mode():
    batch_predictions = model(batch)
print("Input images:", batch.shape[0])
print("Prediction rows:", batch_predictions.shape[0])

Input images: 4
Prediction rows: 4


**Interpretation:** The model returns one prediction row per input image. Batching is useful for throughput; for an interactive app processing one request at a time, single-image latency may matter more.

## 6. CPU vs. GPU inference

**Explanation:** A CPU is commonly available and works well for small workloads. A GPU can process many model operations in parallel and may be faster for larger workloads, but transferring data and starting GPU work also takes time. A GPU is not automatically faster for every small prediction.

**Example:** The setup cell selects CUDA when available; the timing above measures the model on the selected device.

In [6]:
print("Selected device:", device)
if not torch.cuda.is_available():
    print("CUDA is unavailable here; this notebook is using CPU.")
else:
    print("CUDA is available. Compare timing on the same input and batch size before choosing.")

Selected device: cpu
CUDA is unavailable here; this notebook is using CPU.


**Interpretation:** This code reports which device is available, rather than claiming that one device is always best. Compare using the same model, inputs, batch size, and warm-up approach.

## 7. Quantization introduction

**Explanation:** Quantization stores or computes model values with lower numerical precision, such as 8-bit integers instead of 32-bit floats. It can reduce storage and sometimes speed up inference, but may affect accuracy. The actual benefit depends on the model, hardware, and quantization method.

**Example:** Compare the idealized storage for one million values. This is arithmetic only—not a quantized version of our model.

In [7]:
values = 1_000_000
float32_mb = values * 4 / (1024 ** 2)
int8_mb = values * 1 / (1024 ** 2)
print(f"Float32 values: about {float32_mb:.2f} MB")
print(f"8-bit values:   about {int8_mb:.2f} MB")

Float32 values: about 3.81 MB
8-bit values:   about 0.95 MB


**Interpretation:** One byte per value uses about a quarter of the raw storage of four bytes per value. Real model files also have extra data, and quantization must be tested for accuracy and speed on the target hardware.

## 8. ONNX introduction

**Explanation:** ONNX (Open Neural Network Exchange) is a format for representing a model so it can be used with supported tools and runtimes outside its original training framework. Exporting to ONNX does not automatically make a model smaller or faster; measure it in the intended runtime.

**Example export code:** Run this optional snippet only when the ONNX export dependencies are installed. It writes an ONNX file under `artifacts/`.

In [12]:
# Optional: export the loaded PyTorch model for an ONNX-compatible runtime.
# Install the required ONNX exporter dependencies before running this cell.
model_cpu = model.cpu().eval()
example_input = torch.rand(1, 3, 96, 96)
torch.onnx.export(model_cpu, example_input, "artifacts/resnet18_cifar10.onnx",
input_names=["image"], output_names=["scores"],
dynamo=False)

C:\Users\hemak\AppData\Local\Temp\ipykernel_15204\412148267.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model_cpu, example_input, "artifacts/resnet18_cifar10.onnx",


**Interpretation:** The snippet shows the basic export call and input/output names. It is commented out so the notebook runs without optional ONNX packages or creating an extra file.

## 9. Model serialization

**Explanation:** Serialization saves a model so it can be loaded later. A common PyTorch approach is to save the model's `state_dict` (its learned weights) and recreate the same architecture when loading.

**Example:** Save and reload the current weights in a temporary file.

In [13]:
from tempfile import TemporaryDirectory
with TemporaryDirectory() as temp_dir:
    saved_weights = Path(temp_dir) / "model_weights.pth"
    torch.save(model.cpu().state_dict(), saved_weights)
    restored_model = models.resnet18(weights=None)
    restored_model.fc = torch.nn.Linear(restored_model.fc.in_features, 10)
    restored_model.load_state_dict(torch.load(saved_weights, map_location="cpu", weights_only=True))
    print("Saved and reloaded weights:", saved_weights.stat().st_size, "bytes")
model = model.to(device).eval()

Saved and reloaded weights: 44804811 bytes


**Interpretation:** The temporary file shows that weights can be saved and restored without leaving another checkpoint in the project. To load a `state_dict`, the matching model architecture must be created first.

## 10. Accuracy vs. model size vs. inference speed

**Explanation:** These are separate measurements. Accuracy describes prediction quality on an evaluation set; model size describes storage; inference speed describes response time or throughput. Optimization often involves trade-offs, and the best model depends on the deployment goal.

**Example comparison:** The table below is a qualitative illustration, not measured results from this project.

In [14]:
comparison = [
    ("Small model", "may be lower", "usually smaller", "may be faster"),
    ("Large model", "may be higher", "usually larger", "may be slower"),
    ("Optimized model", "must be measured", "may be smaller", "may be faster"),
]
print("Model        | Accuracy       | Size           | Speed")
for name, accuracy, size, speed in comparison:
    print(f"{name:13}| {accuracy:15}| {size:15}| {speed}")

Model        | Accuracy       | Size           | Speed
Small model  | may be lower   | usually smaller| may be faster
Large model  | may be higher  | usually larger | may be slower
Optimized model| must be measured| may be smaller | may be faster


**Interpretation:** A larger model may improve accuracy, but it can require more storage, memory, and time per prediction. An optimized smaller model may be more practical for a phone, edge device, or low-latency service. Measure accuracy on representative data and speed on the target device before choosing.


Before deployment, check model accuracy, parameter count, file size, inference time, and memory needs. Use batches when throughput matters, compare CPU and GPU on the target workload, and evaluate quantization or ONNX in the intended runtime. **The largest model is not always the most practical:** choose one that meets the accuracy requirement while fitting the storage, hardware, and response-time constraints.